# 04: Experiment Graphs

This notebook demonstrates the `ExperimentGraph` class for pipeline-level analysis.

**What you'll learn:**
- Getting a graph from any experiment in a pipeline
- Navigating the graph: `roots`, `leaves`, `experiments`
- Filtering experiments within a graph
- Graph-level artifact loading and parameter/metric access
- The fan-out/collect pattern for HPO and k-fold results

## Prerequisites

You need a pipeline of experiments with dependencies. Run the dependency examples first:

```bash
cd examples/cli/10_dependencies

# Step 1: Preprocess
yanex run prepare_data.py -p dataset=mnist -p samples=1000 --name "prep-v1"
# Note the experiment ID, e.g., abc12345

# Step 2: Train multiple models (fan-out)
yanex run train_model.py -D data=<prep-id> -p "learning_rate=0.001,0.01,0.1" --name "train"

# Step 3: Evaluate each model
yanex run evaluate_model.py -D "model=<train-id-1>,<train-id-2>,<train-id-3>" --name "eval"
```

Replace `<prep-id>` and `<train-id-*>` with actual experiment IDs from step 1 and 2.

In [ ]:
import yanex.results as yr

## Getting a Graph

Start from any experiment ID in the pipeline. By default, the graph returns the **causal lineage** — upstream dependencies and downstream dependents of the starting experiment. Use `weakly_connected=True` to include sibling branches (the full weakly connected component).

In [ ]:
# Replace with an actual experiment ID from your pipeline
EXPERIMENT_ID = "abc12345"

# Default: upstream + downstream lineage only
graph = yr.get_graph(EXPERIMENT_ID)
print(graph)  # ExperimentGraph(experiments=N, edges=M)

# Include sibling branches (full weakly connected component)
full_graph = yr.get_graph(EXPERIMENT_ID, weakly_connected=True)
print(full_graph)

# You can also get a graph from an Experiment object
exp = yr.get_experiment(EXPERIMENT_ID)
graph = exp.get_graph()

## Navigating the Graph

The graph provides properties for navigating the pipeline structure.

In [ ]:
# All experiments in the graph
print(f"Total experiments: {len(graph)}")
for exp in graph:
    print(f"  {exp.id}: {exp.name or '(unnamed)'} [{exp.status}]")

In [ ]:
# Root experiments (no dependencies — e.g., preprocessing)
print("Roots (pipeline entry points):")
for exp in graph.roots:
    print(f"  {exp.id}: {exp.name}")

# Leaf experiments (no dependents — e.g., evaluation)
print("\nLeaves (pipeline endpoints):")
for exp in graph.leaves:
    print(f"  {exp.id}: {exp.name}")

In [ ]:
# Container protocol
print(f"Number of experiments: {len(graph)}")
print(f"Contains {EXPERIMENT_ID}: {EXPERIMENT_ID in graph}")

# Access by ID
exp = graph[EXPERIMENT_ID]
print(f"Accessed: {exp.name}")

## Filtering Experiments in the Graph

Use `graph.filter()` with the same kwargs as `yr.get_experiments()` — `status`, `tags`, `name`, `script_pattern`, etc.

In [ ]:
# Filter by script pattern
train_runs = graph.filter(script_pattern="train_model.py")
print(f"Training runs: {len(train_runs)}")
for run in train_runs:
    lr = run.get_param("learning_rate")
    print(f"  {run.id}: lr={lr}")

In [ ]:
# Filter by status
completed = graph.filter(status="completed")
print(f"Completed: {len(completed)} / {len(graph)}")

# Combine filters
completed_evals = graph.filter(status="completed", script_pattern="evaluate*")
print(f"Completed evaluations: {len(completed_evals)}")

## Graph-Level Data Access

Search all experiments for unique artifacts, parameters, or metrics.

- If found in exactly one experiment (or all have the same value): returns the value
- If found in multiple with different values: raises an error (be more specific)

In [ ]:
# Load an artifact that exists in only one experiment
# (e.g., processed_data.pkl only exists in the preprocessing step)
data = graph.load_artifact("processed_data.pkl")
if data is not None:
    print(f"Loaded processed_data.pkl: {type(data)}")
else:
    print("processed_data.pkl not found in any experiment")

In [ ]:
# Get a parameter that's unique across the graph
dataset = graph.get_param("dataset", default="unknown")
print(f"Dataset: {dataset}")

# Parameters shared with the same value are NOT ambiguous
samples = graph.get_param("samples", default="unknown")
print(f"Samples: {samples}")

In [ ]:
# This would raise ValueError because learning_rate differs across training runs:
try:
    lr = graph.get_param("learning_rate")
except ValueError as e:
    print(f"Expected ambiguity error: {e}")
    print("\nUse graph.filter() to select specific experiments instead.")

## Fan-Out / Collect Pattern

When a pipeline fans out (e.g., HPO sweep, k-fold CV), graph-level accessors will raise
ambiguity errors for duplicated artifacts/params. Use `filter()` + per-experiment access instead.

This is the recommended pattern for collecting results from fan-out pipelines.

In [ ]:
# Collect results from all training runs
print("Training results:")
print(f"{'ID':>10} {'LR':>10} {'Accuracy':>10}")
print("-" * 35)

for run in graph.filter(script_pattern="train_model.py"):
    lr = run.get_param("learning_rate")
    acc = run.get_metric("accuracy")
    print(f"{run.id:>10} {lr:>10} {acc or 'N/A':>10}")

In [ ]:
# Find the best training run
train_runs = graph.filter(script_pattern="train_model.py")

if train_runs:
    best = max(train_runs, key=lambda e: e.get_metric("accuracy") or 0)
    print(f"Best training run: {best.id}")
    print(f"  Learning rate: {best.get_param('learning_rate')}")
    print(f"  Accuracy: {best.get_metric('accuracy')}")

    # Load the best model
    model = best.load_artifact("trained_model.pkl")
    if model:
        print("  Loaded model artifact")

In [ ]:
# Build a results summary dict
results = {}
for run in graph.filter(script_pattern="evaluate*"):
    results[run.id] = {
        "test_accuracy": run.get_metric("test_accuracy"),
        "test_loss": run.get_metric("test_loss"),
    }

print(f"Collected {len(results)} evaluation results")
for exp_id, metrics in results.items():
    print(f"  {exp_id}: acc={metrics['test_accuracy']}, loss={metrics['test_loss']}")

## NetworkX Escape Hatch

For advanced graph analysis, access the underlying `nx.DiGraph` via `graph.digraph`.
Edges point in the data-flow direction (from dependency to dependent).

In [ ]:
import networkx as nx

G = graph.digraph
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

# Topological sort (dependency order)
for node in nx.topological_sort(G):
    exp = graph[node]
    print(f"  {exp.id}: {exp.name or '(unnamed)'}")

## Summary

| What you want | How to do it |
|---|---|
| Get the causal lineage | `graph = yr.get_graph(any_id)` |
| Get full pipeline (incl. siblings) | `graph = yr.get_graph(any_id, weakly_connected=True)` |
| Find pipeline entry points | `graph.roots` |
| Find pipeline endpoints | `graph.leaves` |
| Get a unique artifact | `graph.load_artifact("data.json")` |
| Get a unique parameter | `graph.get_param("dataset")` |
| Collect fan-out results | `graph.filter(script_pattern="train*")` + per-experiment access |
| Find the best run | `max(graph.filter(...), key=lambda e: e.get_metric(...))` |
| Advanced graph analysis | `graph.digraph` (NetworkX DiGraph) |